# Machine Learning

Dieses Notebook dient als Vorbereitung für das spätere .py oder ähnliches für das Dashboard.

## Bibliotheken

In [11]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

## 1. Daten einlesen

In [12]:
data_path = Path("../../Model_data")

data = np.load(data_path / "test_split1_100WS.npz")
X_test_raw = data["X"]
y_test_raw = data["y"]

print(f"X_test_raw.shape: {X_test_raw.shape},\ny_test_raw.shape: {y_test_raw.shape}")
print(f"X_test_raw.dtype: {X_test_raw.dtype},\ny_test_raw.dtype: {y_test_raw.dtype}")


X_test_raw.shape: (3969, 100, 13),
y_test_raw.shape: (3969,)
X_test_raw.dtype: float64,
y_test_raw.dtype: <U9


## 2. Model Artefakte einlesen

In [13]:
model_path = Path("../../Model_data/ML_Daten/final_model_random_forest_100WS.joblib")
artifact = joblib.load(model_path)
artifact.keys(), type(artifact["model"]), type(artifact["scaler"]), type(artifact["label_encoder"])

(dict_keys(['model', 'scaler', 'label_encoder', 'class_names', 'best_params']),
 sklearn.ensemble._forest.RandomForestClassifier,
 sklearn.preprocessing._data.StandardScaler,
 sklearn.preprocessing._label.LabelEncoder)

## 7. Predict Funktion für das .py später

In [14]:
def predict_windows_block(X_windows, artifact):
    model = artifact["model"]
    scaler = artifact["scaler"]
    le = artifact["label_encoder"]

    X_feat = build_feature_matrix(X_windows)
    X_scaled = scaler.transform(X_feat)
    y_pred = le.inverse_transform(model.predict(X_scaled))

    vc = pd.Series(y_pred).value_counts()
    majority = vc.idxmax()
    distribution = (vc / vc.sum()).round(3)

    return {
        "predictions": y_pred,
        "majority_vote": majority,
        "distribution": distribution,
        "n_windows": len(y_pred),
    }

# Diese Zelle kann später in ein .py kopiert werden
Vorausgesetzt es bleibt dabei was zu oberst steht, dass die Daten das selbe Format wie die train.npz haben

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

# ---- Konfiguration ----
npz_path = Path("../../Model_data/test_split1_100WS.npz")  # später: neue Datei
model_path = Path("../../Model_data/ML_Daten/final_model_random_forest_100WS.joblib")

# ---- Feature Engineering (muss identisch zu Training sein) ----
def _extract_channel_features(channel: np.ndarray) -> list:
    mean = np.mean(channel)
    std = np.std(channel)
    min_val = np.min(channel)
    max_val = np.max(channel)
    median = np.median(channel)
    iqr = np.percentile(channel, 75) - np.percentile(channel, 25)

    rms = np.sqrt(np.mean(channel ** 2))
    signal_range = max_val - min_val
    diffs = np.diff(channel)
    mean_abs_diff = np.mean(np.abs(diffs))
    zero_crossings = np.sum(np.diff(np.sign(channel - mean)) != 0)

    t = np.arange(len(channel), dtype=np.float64)
    slope = np.polyfit(t, channel, 1)[0] if std > 1e-10 else 0.0

    return [mean, std, min_val, max_val, median, iqr,
            rms, signal_range, mean_abs_diff, zero_crossings, slope]

def _extract_window_features(window: np.ndarray) -> np.ndarray:
    feats = []
    for ch_idx in range(window.shape[1]):
        feats.extend(_extract_channel_features(window[:, ch_idx]))

    acc_mag = np.sqrt(window[:, 0]**2 + window[:, 1]**2 + window[:, 2]**2)
    feats.extend(_extract_channel_features(acc_mag))

    gyr_mag = np.sqrt(window[:, 3]**2 + window[:, 4]**2 + window[:, 5]**2)
    feats.extend(_extract_channel_features(gyr_mag))

    return np.array(feats, dtype=np.float64)

def build_feature_matrix(X_windows: np.ndarray) -> np.ndarray:
    n_samples = X_windows.shape[0]
    sample = _extract_window_features(X_windows[0])
    X_feat = np.empty((n_samples, len(sample)), dtype=np.float64)
    X_feat[0] = sample
    for i in range(1, n_samples):
        X_feat[i] = _extract_window_features(X_windows[i])
    return X_feat

# ---- Laden ----
data = np.load(npz_path)
X = data["X"]  # erwartet: (n_windows, 100, 13)

artifact = joblib.load(model_path)
model = artifact["model"]
scaler = artifact["scaler"]
le = artifact["label_encoder"]

# ---- Predict ----
X_feat = build_feature_matrix(X)
X_scaled = scaler.transform(X_feat)

# Probability statt Majority Vote
proba = model.predict_proba(X_scaled)
mean_proba = proba.mean(axis=0)
majority = le.inverse_transform([mean_proba.argmax()])[0]
confidence = mean_proba.max()

print(f"{majority} ({confidence:.0%})")

Auto (26%)
